## 프롬프트 엔지니어링 (Prompt Engineering)

프롬프트 엔지니어링은 단순히 질문을 던지는 것을 넘어, 모델의 작동 원리와 ‘인컨텍스트 러닝(In-context Learning)’ 능력을 활용해 모델의 출력을 제어하는 프로세스이다. 이는 모델의 파라미터(가중치)를 직접 수정하지 않고도 모델의 성능을 특정 태스크에 맞게 조정하는 방법론이다.

### 프롬프트 핵심 구성 요소

효과적인 프롬프트는 일반적으로 다음의 4가지 요소를 포함한다.

- 지시문 (Instruction): 모델이 수행해야 할 구체적인 작업 (예: 요약하라, 분류하라, 번역하라 등)
- 문맥 (Context): 모델이 작업을 더 잘 수행하도록 돕는 배경 정보나 제약 조건
- 입력 데이터 (Input Data): 처리가 필요한 실제 데이터
- 출력 지시자 (Output Indicator): 결과물의 형식이나 스타일 지정 (예: 표로 정리하라, JSON 포맷으로 출력하라 등)

### 프롬프트 엔지니어링의 중요성

- 성능 최적화: 같은 모델이라도 프롬프트에 따라 성능 차이가 극심하다. 잘 설계된 프롬프트는 더 작은 모델로도 큰 모델 수준의 결과를 낼 수 있게 한다.
- 비용 효율성: 불필요한 토큰 사용을 줄이고, 파인튜닝(Fine-tuning)에 비해 적은 비용으로 도메인 특화 작업을 수행할 수 있다.
- 한계 극복: 모델의 환각 현상을 줄이고 최신 정보를 반영(RAG와 결합)하더라도 유도할 수 있다.

## 환경설정
- OpenAI에서 api key 발급 받아 .env 파일에 저장 (api key 노출 되지 않도록 유의)
- 환경 변수 로드 가능한 `python-dotenv`와 api 호출을 위한 `openai` 설치

In [1]:
%pip install openai python-dotenv

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 7.5 MB/s  0:00:00

   -------- ------------------------------- 1/5 [python-dotenv]
   -------- ------------------------------- 1/5 [python-dotenv]
   ------------------------ --------------- 3/5 [distro]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# 다수의 api를 모두 호출할 할 수 있는 client 객체
client = OpenAI()

## 기사 제목 교정 

In [2]:
response = client.chat.completions.create(
  model="gpt-4.1-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "# SYSTEM PROMPT\n\n당신은 기사 제목을 교정하는 전문가입니다. 입력으로 주어진 기사 제목의 맞춤법, 띄어쓰기, 문법, 표현이 적절한지 세밀하게 검토한 후, 올바르고 자연스러운 형태로 교정하십시오. 결론(교정된 제목)을 제시하기 전에, 반드시 내부적으로 교정 요소(오류 및 개선할 점)에 대한 체계적인 판단 과정을 거친 뒤, 교정된 제목을 마지막에 출력하십시오.\n\n- 아래의 순서로 답변을 구성하세요:\n  1. **교정 요소 분석(Reasoning)**: 주어진 제목에서 발견된 오류, 어색함, 또는 개선점을 구체적으로 서술하십시오.\n  2. **교정된 제목(Conclusion)**: 모든 분석·판단이 끝난 후에, 교정된 제목을 단독으로 한 줄에 제시하세요.\n\n- 답변 형식: 아래 예시를 따르세요.\n\n---\n\n### 예시\n\n#### 입력 제목:\n정부, 경제 성장률 상향 전망\n\n#### 출력:\n- 교정 요소 분석:\n  - '상향 전망'은 애매한 표현으로, '상향 조정'이나 '상향 발표'가 더 명확합니다.\n  - 전반적으로 문법 오류는 없으나, 의미 전달을 명확히 하기 위해 수정이 필요합니다.\n- 교정된 제목:\n  정부, 경제 성장률 상향 조정 발표\n\n---\n\n### 추가 안내/고려 사항\n\n- 반드시 **분석 → 교정 결과** 순서로 답변하세요.\n- 교정된 제목은 명확하고 간결하게 표현하세요.\n- 형식은 마크다운 리스트(- 혹은 번호)로만 작성하세요.\n- 내용상 부적절하거나 모욕적인 제목은 반드시 수정하세요.\n- 실제 답변 예시는 위 예시 수준처럼 간결하되, 실제 상황에서는 더 길거나 복잡한 제목도 다뤄야 하므로 충분한 표현력과 분석력을 보여주십시오.\n\n---\n\n**반드시:**   \n입력된 기사 제목의 교정 요소를 먼저 '분석'하여 서술한 후, 마지막에 '교정된 제목'을 명확히 한 문장으로 제시하세요."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "입력 제목:\n어제 서울에서의 큰 불이나서 수백명이 대피했다."
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "- 교정 요소 분석:  \n  - '불이나서'는 '불이 나서'로 띄어 써야 하는 합성어 구입니다.  \n  - '수백명이'에서 '명이'는 띄어 써야 하므로 '수백 명이'로 수정해야 합니다.  \n  - '서울에서의'는 '에서'와 '의'가 중복되어 어색하므로, 자연스럽게 '서울에서'로 고치는 것이 좋습니다.  \n  - 전반적으로 문장 끝에 마침표는 적절하나, 보다 간결하고 자연스러운 표현을 위해 문장 구조를 약간 다듬을 수 있습니다.  \n\n- 교정된 제목:  \n  어제 서울에서 큰 불이 나 수백 명이 대피했다."
        }
      ]
    }
  ],
  response_format={
    "type": "text"
  },
  temperature=1,
  max_completion_tokens=2048,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0,
  store=False
)

In [3]:
response

ChatCompletion(id='chatcmpl-DcP3Uv63L5reP0nS2uUe9btdVjxUk', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="- 교정 요소 분석:  \n  - '불이나서'는 '불이 나서'로 띄어 써야 하며, 맞춤법에 맞지 않습니다.  \n  - '수백명이'는 수량 단위 뒤에 조사 '명이'가 붙는 경우, '수백 명이'로 띄어 써야 합니다.  \n  - '서울에서의'는 '서울에서'로 해도 충분히 의미 전달이 되며, '에서의'가 과하게 느껴져 불필요한 표현입니다.  \n  - 전반적으로 문장이 자연스럽고 명확하게 전달되도록 다듬는 것이 좋습니다.  \n\n- 교정된 제목:  \n  어제 서울에서 큰 불이 나 수백 명이 대피했다.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1778045148, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_81d3e80949', usage=CompletionUsage(completion_tokens=169, prompt_tokens=688, total_tokens=857, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [4]:
response.choices[0].message.content

"- 교정 요소 분석:  \n  - '불이나서'는 '불이 나서'로 띄어 써야 하며, 맞춤법에 맞지 않습니다.  \n  - '수백명이'는 수량 단위 뒤에 조사 '명이'가 붙는 경우, '수백 명이'로 띄어 써야 합니다.  \n  - '서울에서의'는 '서울에서'로 해도 충분히 의미 전달이 되며, '에서의'가 과하게 느껴져 불필요한 표현입니다.  \n  - 전반적으로 문장이 자연스럽고 명확하게 전달되도록 다듬는 것이 좋습니다.  \n\n- 교정된 제목:  \n  어제 서울에서 큰 불이 나 수백 명이 대피했다."

In [5]:
# input data 수정 테스트
response = client.chat.completions.create(
  model="gpt-4.1-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "# SYSTEM PROMPT\n\n당신은 기사 제목을 교정하는 전문가입니다. 입력으로 주어진 기사 제목의 맞춤법, 띄어쓰기, 문법, 표현이 적절한지 세밀하게 검토한 후, 올바르고 자연스러운 형태로 교정하십시오. 결론(교정된 제목)을 제시하기 전에, 반드시 내부적으로 교정 요소(오류 및 개선할 점)에 대한 체계적인 판단 과정을 거친 뒤, 교정된 제목을 마지막에 출력하십시오.\n\n- 아래의 순서로 답변을 구성하세요:\n  1. **교정 요소 분석(Reasoning)**: 주어진 제목에서 발견된 오류, 어색함, 또는 개선점을 구체적으로 서술하십시오.\n  2. **교정된 제목(Conclusion)**: 모든 분석·판단이 끝난 후에, 교정된 제목을 단독으로 한 줄에 제시하세요.\n\n- 답변 형식: 아래 예시를 따르세요.\n\n---\n\n### 예시\n\n#### 입력 제목:\n정부, 경제 성장률 상향 전망\n\n#### 출력:\n- 교정 요소 분석:\n  - '상향 전망'은 애매한 표현으로, '상향 조정'이나 '상향 발표'가 더 명확합니다.\n  - 전반적으로 문법 오류는 없으나, 의미 전달을 명확히 하기 위해 수정이 필요합니다.\n- 교정된 제목:\n  정부, 경제 성장률 상향 조정 발표\n\n---\n\n### 추가 안내/고려 사항\n\n- 반드시 **분석 → 교정 결과** 순서로 답변하세요.\n- 교정된 제목은 명확하고 간결하게 표현하세요.\n- 형식은 마크다운 리스트(- 혹은 번호)로만 작성하세요.\n- 내용상 부적절하거나 모욕적인 제목은 반드시 수정하세요.\n- 실제 답변 예시는 위 예시 수준처럼 간결하되, 실제 상황에서는 더 길거나 복잡한 제목도 다뤄야 하므로 충분한 표현력과 분석력을 보여주십시오.\n\n---\n\n**반드시:**   \n입력된 기사 제목의 교정 요소를 먼저 '분석'하여 서술한 후, 마지막에 '교정된 제목'을 명확히 한 문장으로 제시하세요."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "입력 제목:\n두쫀쿠 열풍 뭐가 존맛탱이라는지 알 수가 없네"
        }
      ]
    }
  ],
  response_format={
    "type": "text"
  },
  temperature=1,
  max_completion_tokens=2048,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0,
  store=False
)

In [6]:
response.choices[0].message.content

"- 교정 요소 분석:  \n  - '두쫀쿠'는 대중적으로 통용되는 단어인지 불분명해, 독자의 이해를 돕기 위해 단어 설명이나 정식 명칭 사용을 고려할 수 있습니다. 다만 고유명사나 브랜드명일 수 있으므로 그대로 둘 수 있습니다.  \n  - '뭐가'는 구어체 표현이며, 기사 제목에는 다소 비격식적입니다. '무엇이'로 고치면 더 공식적이고 정제된 느낌이 납니다.  \n  - '존맛탱'은 속어로 '정말 맛있다'는 의미이나, 기사 제목에서는 비공식적이고 청소년어가 포함돼 있어 부적절할 수 있습니다. 대신 '정말 맛있다는 것' 등으로 바꾸는 것이 적절합니다.  \n  - '이라는지'는 연결어미로 자연스럽지만, 전체 문장 구성을 다듬어 더 명확하고 간결한 문장으로 교체할 필요가 있습니다.  \n  - '알 수가 없네'는 다소 구어체이므로 '이해하기 어렵다' 또는 '이해할 수 없다'로 교체하면 더 격식에 맞습니다.  \n  - 전반적으로 구어체와 속어가 섞여 있어 뉴스 기사 제목으로서 적합하지 않습니다.\n\n- 교정된 제목:  \n두쫀쿠 열풍, 무엇이 그렇게 맛있다는 것인지 이해하기 어렵다"

In [1]:
# 재사용을 위한 함수 선언
def correct_headline(headline, model="gpt-4.1-mini", temperature=1, max_completion_tokens=2048, top_p=1):
  system_message = "# SYSTEM PROMPT\n\n당신은 기사 제목을 교정하는 전문가입니다. 입력으로 주어진 기사 제목의 맞춤법, 띄어쓰기, 문법, 표현이 적절한지 세밀하게 검토한 후, 올바르고 자연스러운 형태로 교정하십시오. 결론(교정된 제목)을 제시하기 전에, 반드시 내부적으로 교정 요소(오류 및 개선할 점)에 대한 체계적인 판단 과정을 거친 뒤, 교정된 제목을 마지막에 출력하십시오.\n\n- 아래의 순서로 답변을 구성하세요:\n  1. **교정 요소 분석(Reasoning)**: 주어진 제목에서 발견된 오류, 어색함, 또는 개선점을 구체적으로 서술하십시오.\n  2. **교정된 제목(Conclusion)**: 모든 분석·판단이 끝난 후에, 교정된 제목을 단독으로 한 줄에 제시하세요.\n\n- 답변 형식: 아래 예시를 따르세요.\n\n---\n\n### 예시\n\n#### 입력 제목:\n정부, 경제 성장률 상향 전망\n\n#### 출력:\n- 교정 요소 분석:\n  - '상향 전망'은 애매한 표현으로, '상향 조정'이나 '상향 발표'가 더 명확합니다.\n  - 전반적으로 문법 오류는 없으나, 의미 전달을 명확히 하기 위해 수정이 필요합니다.\n- 교정된 제목:\n  정부, 경제 성장률 상향 조정 발표\n\n---\n\n### 추가 안내/고려 사항\n\n- 반드시 **분석 → 교정 결과** 순서로 답변하세요.\n- 교정된 제목은 명확하고 간결하게 표현하세요.\n- 형식은 마크다운 리스트(- 혹은 번호)로만 작성하세요.\n- 내용상 부적절하거나 모욕적인 제목은 반드시 수정하세요.\n- 실제 답변 예시는 위 예시 수준처럼 간결하되, 실제 상황에서는 더 길거나 복잡한 제목도 다뤄야 하므로 충분한 표현력과 분석력을 보여주십시오.\n\n---\n\n**반드시:**   \n입력된 기사 제목의 교정 요소를 먼저 '분석'하여 서술한 후, 마지막에 '교정된 제목'을 명확히 한 문장으로 제시하세요."
  user_message = f"입력 제목:\n{headline}"
  response = client.chat.completions.create(
    model=model,
    messages=[
      {
        "role": "system",
        "content": [
          {
            "type": "text",
            "text": system_message
          }
        ]
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": user_message
          }
        ]
      }
    ],
    response_format={
      "type": "text"
    },
    temperature=temperature,
    max_completion_tokens=max_completion_tokens,
    top_p=top_p,
    frequency_penalty=0,
    presence_penalty=0,
    store=False
  )
  return response.choices[0].message.content

In [4]:
correct_headline('두쫀쿠 열풍 뭐가 존맛탱이라는지 알 수가 없네', model='gpt-5.4')

"- 교정 요소 분석:\n  - '두쫀쿠'는 일반적으로 통용되는 표현이 아니며, 제목만으로는 의미가 불분명합니다. 다만 고유명사나 유행어일 가능성이 있어 임의로 바꾸기보다는 원형을 유지하는 것이 적절합니다.\n  - '뭐가'는 구어체적 표현으로, 기사 제목에서는 '무엇이'로 다듬는 것이 더 자연스럽습니다.\n  - '존맛탱'은 비속어·신조어로, 기사 제목에 사용하기에는 부적절하므로 '그렇게 맛있다는지' 또는 '인기라는지' 등 중립적 표현으로 순화할 필요가 있습니다.\n  - '알 수가 없네'는 일상 대화체 종결 표현이므로, 기사 제목에 맞게 '알 수 없어', '이해하기 어려워' 등으로 정제하는 것이 좋습니다.\n  - 전체적으로 감정적인 구어체가 강하므로, 기사 제목에 맞는 객관적이고 자연스러운 문장으로 조정하는 것이 필요합니다.\n- 교정된 제목:\n  두쫀쿠 열풍, 무엇이 그렇게 맛있다는 건지 알 수 없어"

In [5]:
# 여러 건의 제목 수정
headlines = [
    '주말 성수동 거리에는 커플들이 그득그득 하더라',
    '유투버의 생존일기, 쉽지 않은 그들의 여정에 함께해요~',
    '빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환'
]

for headline in headlines:
    print(correct_headline(headline))
    print()

- 교정 요소 분석:
  - '그득그득 하더라'는 구어체 표현이며, '그득그득하다'가 올바른 형태이나 보통은 '가득하다' 또는 '붐비다' 같은 자연스러운 표현을 사용합니다.
  - '그득그득' 자체가 비표준어로 오히려 부자연스러우므로, 보다 표준적이고 자연스러운 부사로 교체하는 것이 좋습니다.
  - '커플들'은 복수 명사로 적절하며 띄어쓰기 및 맞춤법에 문제 없습니다.
  - 전체 문장이 구어체 서술 방식(감탄적, 회고적 느낌)으로 되어 있어 기사 제목으로는 다소 비격식적입니다.
  - 기사 제목의 성격을 살리면서 표현을 간결하고 표준어 위주로 교정하는 것이 바람직합니다.
- 교정된 제목:
주말 성수동 거리에 커플들이 가득하다

- 교정 요소 분석:  
  - '유투버'는 국립국어원 표준 표기인 '유튜버'로 수정하는 것이 맞습니다.  
  - '쉽지 않은 그들의 여정에 함께해요~'에서 '~'는 기사 제목에서는 다소 비격식적이고 부적절하므로 삭제하는 것이 좋습니다.  
  - 전체적으로 친근한 뉘앙스를 유지하되, 기사 제목이므로 어느 정도 공식적이고 간결하게 수정하는 게 바람직합니다.  
  - '함께해요'도 친근한 표현이나, 기사 제목으로는 다소 구어체로 느껴질 수 있으므로 '함께하는' 등으로 변경해도 좋으나, 상황에 따라 유지 가능함.  
  - 쉼표 대신 '–'나 마침표로 구분하거나 쉼표를 유지해도 큰 문제는 없으나, 쉼표 사용은 자연스러움.  

- 교정된 제목:  
  유튜버의 생존일기, 쉽지 않은 그들의 여정에 함께해요

- 교정 요소 분석:  
  - '빡센'은 구어체 속어로, 공식 뉴스 기사 제목에 적합하지 않습니다. '힘든', '고된' 등으로 교체하는 것이 좋습니다.  
  - '끼니 거르기'는 다소 어색한 표현으로, '끼니를 거르기'가 올바른 띄어쓰기입니다.  
  - '일쑤인'도 구어체 느낌이 강하며, '일쑤인' 자체는 크게 문제 없으나 좀 더 표준어에 가까운 표현으로 바꾸는 것도 고려할 수 있습니다. 다만 큰 문제는 아니므로 유지해도 무방

## JSON Object 반환으로 변경

기자들이 송고한 제목에서 맞춤법/문법/의미/어조 등을 고려해 최상의 뉴스 제목을 뽑아내는 20년 경력의 뉴스데스크장입니다.

## Instruction
교정이 필요한 기사 제목을 입력받아 맞춤법, 띄어쓰기, 문법, 의미, 어조를 점검하고 더 나은 뉴스 제목으로 교정하세요.

아래 기준에 따라 작업합니다.

1. 입력된 기사 제목을 분석하여 맞춤법 오류, 띄어쓰기 오류, 문법 오류, 의미상 어색한 표현, 과도하게 감정적이거나 부정적인 표현을 찾습니다.
2. 오류가 있다면 모두 반영하여 간결하고 명확한 뉴스 제목으로 수정합니다.
3. 비속어/욕설이 포함되어 있다면 제거하고, 의미가 전달되는 중립적 표현으로 수정합니다.
4. 오류가 여러 개 있을 경우 각각의 교정 이유를 구분하여 작성합니다.
5. 오류가 없더라도 더 자연스럽고 기사 제목에 적합한 표현이 있다면 다듬을 수 있습니다.
6. 응답은 반드시 유효한 JSON 객체 하나만 반환합니다.
7. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.

## Output JSON Format
{
  "original_title": "입력된 원제목",
  "corrected_title": "교정된 제목",
  "correction_reasons": [
    {
      "original": "교정 전 표현",
      "corrected": "교정 후 표현",
      "reason": "교정 이유"
    }
  ]
}

## Rules
- original_title에는 사용자가 입력한 제목을 그대로 작성합니다.
- corrected_title에는 최종 교정 제목만 작성합니다.
- correction_reasons는 배열로 작성합니다.
- 교정할 부분이 없다면 correction_reasons는 빈 배열([])로 작성합니다.
- 모든 문자열은 한국어로 작성합니다.

In [6]:
import json

# 재사용을 위한 함수 선언(JSON)
def correct_headline_json(headline, model="gpt-4.1-mini", temperature=1, max_completion_tokens=2048, top_p=1):
  system_message = """
기자들이 송고한 제목에서 맞춤법/문법/의미/어조 등을 고려해 최상의 뉴스 제목을 뽑아내는 20년 경력의 뉴스데스크장입니다.

## Instruction
교정이 필요한 기사 제목을 입력받아 맞춤법, 띄어쓰기, 문법, 의미, 어조를 점검하고 더 나은 뉴스 제목으로 교정하세요.

아래 기준에 따라 작업합니다.

1. 입력된 기사 제목을 분석하여 맞춤법 오류, 띄어쓰기 오류, 문법 오류, 의미상 어색한 표현, 과도하게 감정적이거나 부정적인 표현을 찾습니다.
2. 오류가 있다면 모두 반영하여 간결하고 명확한 뉴스 제목으로 수정합니다.
3. 비속어/욕설이 포함되어 있다면 제거하고, 의미가 전달되는 중립적 표현으로 수정합니다.
4. 오류가 여러 개 있을 경우 각각의 교정 이유를 구분하여 작성합니다.
5. 오류가 없더라도 더 자연스럽고 기사 제목에 적합한 표현이 있다면 다듬을 수 있습니다.
6. 응답은 반드시 유효한 JSON 객체 하나만 반환합니다.
7. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.

## Output JSON Format
{
  "original_title": "입력된 원제목",
  "corrected_title": "교정된 제목",
  "correction_reasons": [
    {
      "original": "교정 전 표현",
      "corrected": "교정 후 표현",
      "reason": "교정 이유"
    }
  ]
}

## Rules
- original_title에는 사용자가 입력한 제목을 그대로 작성합니다.
- corrected_title에는 최종 교정 제목만 작성합니다.
- correction_reasons는 배열로 작성합니다.
- 교정할 부분이 없다면 correction_reasons는 빈 배열([])로 작성합니다.
- 모든 문자열은 한국어로 작성합니다.
    """
  user_message = f"입력 제목:\n{headline}"
  response = client.chat.completions.create(
    model=model,
    messages=[
      {
        "role": "system",
        "content": [
          {
            "type": "text",
            "text": system_message
          }
        ]
      },
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": user_message
          }
        ]
      }
    ],
    response_format={
      "type": "json_object"
    },
    temperature=temperature,
    max_completion_tokens=max_completion_tokens,
    top_p=top_p,
    frequency_penalty=0,
    presence_penalty=0,
    store=False
  )
  return json.loads(response.choices[0].message.content)

In [7]:
# 여러 건의 제목 수정
headlines = [
    '주말 성수동 거리에는 커플들이 그득그득 하더라',
    '유투버의 생존일기, 쉽지 않은 그들의 여정에 함께해요~',
    '빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환'
]

for headline in headlines:
    print(correct_headline_json(headline))
    print()

{'original_title': '주말 성수동 거리에는 커플들이 그득그득 하더라', 'corrected_title': '주말 성수동 거리에는 커플들로 가득했다', 'correction_reasons': [{'original': '커플들이 그득그득 하더라', 'corrected': '커플들로 가득했다', 'reason': '‘그득그득’은 비표준어이며, ‘가득하다’가 더 자연스럽고 정확한 표현입니다. ‘커플들이’는 주어와 부사가 어색하여 ‘커플들로’로 수정해 자연스러운 문장으로 다듬었습니다.'}, {'original': '하더라', 'corrected': '했다', 'reason': '기사 제목은 객관적이고 간결한 어조가 적합하므로 구어체인 ‘하더라’를 ‘했다’로 바꾸어 공식적인 표현으로 수정했습니다.'}]}

{'original_title': '유투버의 생존일기, 쉽지 않은 그들의 여정에 함께해요~', 'corrected_title': '유튜버의 생존 일기, 쉽지 않은 그들의 여정에 함께합니다', 'correction_reasons': [{'original': '유투버', 'corrected': '유튜버', 'reason': '‘유튜버’의 정확한 표기입니다.'}, {'original': '생존일기', 'corrected': '생존 일기', 'reason': '‘생존 일기’는 두 단어로 띄어쓰는 것이 올바른 표현입니다.'}, {'original': '함께해요~', 'corrected': '함께합니다', 'reason': '신문 기사 제목에 어울리는 격식 있고 간결한 표현으로 수정했습니다. 또한 ‘~’ 기호는 공식 문서나 기사 제목에 부적합합니다.'}]}

{'original_title': '빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환', 'corrected_title': '고된 작업으로 끼니를 거르기 일쑤인 노동자들의 애환', 'correction_reasons': [{'original': '빡센 작업', 'corrected': '고된 작업', 'reaso

## 냉털 마스터
- 사용자는 냉장고에 남아있는 음식 재료를 input data로 전달한다.
- 해당 음식 재료를 기반으로 어떤 음식을 만들지 조언(레시피 포함)한다.
- 적절한 프롬프팅을 통해 구현한다.
- 응답 양식은 JSON 형태로 구현한다.

In [11]:
import json

def fridge_raid_master(
    user_foods,
    model="gpt-4.1-mini",
    temperature=0.7,
    max_completion_tokens=2048,
    top_p=1
):
    system_message = """
당신은 냉장고 속 남은 재료를 바탕으로 현실적인 한 끼를 추천하는 요리 도우미입니다.

사용자는 냉장고에 남아 있는 재료 목록을 리스트 형태로 전달합니다.
당신은 입력 재료를 분석하여 실제로 만들기 좋은 요리 1개와 간단한 레시피를 JSON 형식으로 반환해야 합니다.

## 판단 기준

1. 입력 재료 중 자연스럽게 어울리는 조합을 우선 선택합니다.
2. 모든 재료를 억지로 사용하지 않습니다.
3. 향이나 염도가 강한 재료는 다른 재료와 충돌할 경우 제외할 수 있습니다.
   예: 명란젓, 된장, 김치, 고추장, 젓갈류 등
4. 주재료와 부재료를 구분하여 요리를 설계합니다.
5. 초보자도 만들 수 있는 가정식 수준의 요리만 추천합니다.
6. 튀김, 오븐 요리, 장시간 숙성, 전문 장비가 필요한 조리법은 피합니다.
7. 추가 재료는 일반 가정에 있을 가능성이 높은 기본 양념으로 제한합니다.
   예: 소금, 후추, 식용유, 물, 간장, 설탕, 다진 마늘, 고춧가루, 참기름
8. 입력 재료 중 사용하지 않는 재료가 있다면 unused_ingredients에 이유를 작성합니다.
9. 추천 요리는 1개만 제안합니다.
10. 응답은 반드시 유효한 JSON 객체 하나만 반환합니다.
11. JSON 외의 설명, 마크다운, 코드블럭, 주석은 절대 포함하지 않습니다.

## 추천 우선순위

1순위: 입력 재료만으로 자연스럽게 만들 수 있는 요리
2순위: 기본 양념만 추가하면 만들 수 있는 요리
3순위: 일부 재료를 제외하더라도 맛의 완성도가 높은 요리

## Output JSON Format

{
  "input_ingredients": ["사용자가 입력한 재료"],
  "recommended_dish": "추천 요리명",
  "dish_summary": "요리에 대한 한 줄 설명",
  "used_ingredients": ["실제로 사용하는 입력 재료"],
  "additional_ingredients": ["추가로 필요한 기본 재료"],
  "unused_ingredients": [
    {
      "ingredient": "사용하지 않은 재료",
      "reason": "사용하지 않은 이유"
    }
  ],
  "recipe": {
    "servings": "몇 인분인지",
    "estimated_time": "예상 조리 시간",
    "difficulty": "쉬움 | 보통 | 어려움",
    "steps": [
      "1단계 조리 설명",
      "2단계 조리 설명",
      "3단계 조리 설명",
      "4단계 조리 설명"
    ]
  },
  "taste_profile": "맛의 특징",
  "why_recommended": "이 요리를 추천하는 이유",
  "tips": [
    "조리 팁 1",
    "조리 팁 2"
  ]
}

## 출력 규칙

- input_ingredients에는 사용자가 입력한 재료를 그대로 작성합니다.
- used_ingredients에는 입력 재료 중 실제로 사용하는 재료만 작성합니다.
- additional_ingredients에는 입력에 없지만 필요한 기본 재료만 작성합니다.
- 추가 재료가 필요 없다면 additional_ingredients는 빈 배열([])로 작성합니다.
- 사용하지 않는 입력 재료가 없다면 unused_ingredients는 빈 배열([])로 작성합니다.
- recipe.steps는 최소 4단계 이상 작성합니다.
- difficulty는 반드시 "쉬움", "보통", "어려움" 중 하나로 작성합니다.
- 모든 문자열은 한국어로 작성합니다.
"""

    user_message = f"""
다음은 사용자가 냉장고에 가지고 있는 재료입니다.

{user_foods}

위 재료를 바탕으로 실제로 만들기 좋은 요리 1개를 추천하세요.
모든 재료를 억지로 사용하지 말고, 가장 맛이 자연스러운 조합을 선택하세요.
응답은 지정된 JSON 형식으로만 작성하세요.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": system_message
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": user_message
                    }
                ]
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "fridge_recipe_schema",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "input_ingredients": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "recommended_dish": {
                            "type": "string"
                        },
                        "dish_summary": {
                            "type": "string"
                        },
                        "used_ingredients": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "additional_ingredients": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "unused_ingredients": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "ingredient": {"type": "string"},
                                    "reason": {"type": "string"}
                                },
                                "required": ["ingredient", "reason"],
                                "additionalProperties": False
                            }
                        },
                        "recipe": {
                            "type": "object",
                            "properties": {
                                "servings": {"type": "string"},
                                "estimated_time": {"type": "string"},
                                "difficulty": {
                                    "type": "string",
                                    "enum": ["쉬움", "보통", "어려움"]
                                },
                                "steps": {
                                    "type": "array",
                                    "items": {"type": "string"},
                                    "minItems": 4
                                }
                            },
                            "required": [
                                "servings",
                                "estimated_time",
                                "difficulty",
                                "steps"
                            ],
                            "additionalProperties": False
                        },
                        "taste_profile": {
                            "type": "string"
                        },
                        "why_recommended": {
                            "type": "string"
                        },
                        "tips": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "input_ingredients",
                        "recommended_dish",
                        "dish_summary",
                        "used_ingredients",
                        "additional_ingredients",
                        "unused_ingredients",
                        "recipe",
                        "taste_profile",
                        "why_recommended",
                        "tips"
                    ],
                    "additionalProperties": False
                }
            }
        },
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0,
        store=False
    )

    return json.loads(response.choices[0].message.content)

## [참고] JSON Schema 기반 응답 형식 지정

`response_format={"type": "json_schema"}`는 모델의 응답을 정해진 JSON 구조에 맞게 받기 위해 사용한다.

기존의 `json_object`는 “JSON 객체로 응답하라”는 정도만 보장한다.

    response_format={
        "type": "json_object"
    }

하지만 이 방식은 다음을 엄격하게 보장하지는 못한다.

- 필요한 필드가 모두 있는지
- 필드명이 정확한지
- 값의 자료형이 맞는지
- 배열 안의 값 형태가 맞는지
- 정해진 값만 사용했는지

그래서 응답 결과를 코드에서 바로 사용해야 할 때는 `json_schema`를 사용하는 것이 더 안정적이다.

## 기본 구조

    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "fridge_recipe_schema",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "recommended_dish": {
                        "type": "string"
                    }
                },
                "required": ["recommended_dish"],
                "additionalProperties": False
            }
        }
    }

## 주요 작성법

### 1. name

스키마의 이름을 지정한다.

    "name": "fridge_recipe_schema"

### 2. strict

스키마를 엄격하게 따르도록 설정한다.

    "strict": True

### 3. type

값의 자료형을 지정한다.

    "type": "object"

자주 사용하는 타입은 다음과 같다.

    object  : JSON 객체
    array   : 배열
    string  : 문자열
    number  : 숫자
    integer : 정수
    boolean : 참/거짓

### 4. properties

JSON 객체 안에 들어갈 필드를 정의한다.

    "properties": {
        "recommended_dish": {
            "type": "string"
        },
        "dish_summary": {
            "type": "string"
        }
    }

### 5. required

반드시 포함되어야 하는 필드를 지정한다.

    "required": [
        "recommended_dish",
        "dish_summary"
    ]

### 6. array와 items

배열은 `array`, 배열 안의 값은 `items`로 지정한다.

    "input_ingredients": {
        "type": "array",
        "items": {
            "type": "string"
        }
    }

위 설정은 다음과 같은 값을 기대한다.

    "input_ingredients": ["소고기", "양파", "표고버섯"]

### 7. enum

허용할 값을 제한할 때 사용한다.

    "difficulty": {
        "type": "string",
        "enum": ["쉬움", "보통", "어려움"]
    }

위 설정을 사용하면 `difficulty`에는 `"쉬움"`, `"보통"`, `"어려움"` 중 하나만 들어갈 수 있다.

### 8. minItems

배열의 최소 개수를 지정한다.

    "steps": {
        "type": "array",
        "items": {
            "type": "string"
        },
        "minItems": 4
    }

위 설정은 `steps` 배열에 최소 4개의 값이 필요하다는 뜻이다.

### 9. additionalProperties

정의하지 않은 필드를 허용할지 결정한다.

    "additionalProperties": False

`False`로 설정하면 스키마에 없는 필드가 추가되는 것을 막을 수 있다.

## 현재 예제에서의 역할

냉장고 레시피 예제에서는 다음 구조를 일정하게 유지하기 위해 `json_schema`를 사용한다.

    {
        "input_ingredients": [],
        "recommended_dish": "",
        "dish_summary": "",
        "used_ingredients": [],
        "additional_ingredients": [],
        "unused_ingredients": [],
        "recipe": {
            "servings": "",
            "estimated_time": "",
            "difficulty": "",
            "steps": []
        },
        "taste_profile": "",
        "why_recommended": "",
        "tips": []
    }

이렇게 구조를 고정해두면 응답을 받은 뒤 바로 Python 객체로 변환해서 사용하기 쉽다.

In [12]:
user_foods = ['소고기', '양파', '표고버섯', '명란젓', '된장']
fridge_raid_master(user_foods)

{'input_ingredients': ['소고기', '양파', '표고버섯', '명란젓', '된장'],
 'recommended_dish': '소고기 버섯 볶음',
 'dish_summary': '부드러운 소고기와 향긋한 표고버섯, 양파를 간단히 볶아 맛있게 즐기는 한 끼 요리입니다.',
 'used_ingredients': ['소고기', '양파', '표고버섯'],
 'additional_ingredients': ['간장', '소금', '후추', '식용유', '다진 마늘'],
 'unused_ingredients': [{'ingredient': '명란젓',
   'reason': '향과 염도가 강해 소고기와 표고버섯 볶음과 조화가 어렵고 맛의 균형을 해칠 수 있음'},
  {'ingredient': '된장',
   'reason': '된장은 별도의 찌개나 국 요리에 적합하며 볶음 요리에 사용 시 맛이 강해 다른 재료와 어울리기 어려움'}],
 'recipe': {'servings': '2인분',
  'estimated_time': '20분',
  'difficulty': '쉬움',
  'steps': ['소고기는 먹기 좋은 크기로 썰고, 양파와 표고버섯은 채썰어 준비한다.',
   '팬에 식용유를 두르고 다진 마늘을 넣어 향을 낸다.',
   '소고기를 넣고 중불에서 익을 때까지 볶는다.',
   '양파와 표고버섯을 넣고 간장, 소금, 후추로 간을 하며 재료가 부드러워질 때까지 볶는다.']},
 'taste_profile': '고소하고 담백하며 버섯과 양파의 달큰한 풍미가 어우러진 맛',
 'why_recommended': '간단한 재료로도 풍부한 맛을 낼 수 있고, 강한 향신료나 염도가 없는 재료 중심으로 구성해 누구나 쉽게 만들 수 있기 때문입니다.',
 'tips': ['소고기는 너무 오래 볶지 않도록 주의해 부드러운 식감을 유지하세요.',
  '버섯과 양파는 너무 익히면 식감이 무너지므로 적당히 익히는 것이 좋습니다.']}

In [13]:
# 여러 테스트 케이스 준비
test_food_lists = [
    ['계란', '대파', '밥', '김치'],
    ['닭가슴살', '양배추', '당근', '양파'],
    ['참치캔', '김치', '밥', '계란'],
    ['파스타면', '베이컨', '마늘', '양파'],
    ['떡', '어묵', '양배추', '대파'],
]

# 여러 번 테스트 실행
results = []

for idx, foods in enumerate(test_food_lists, start=1):
    print(f"\n===== 테스트 {idx} =====")
    print("입력 재료:", foods)

    result = fridge_raid_master(foods)

    results.append(result)

    print("추천 요리:", result["recommended_dish"])
    print("한 줄 설명:", result["dish_summary"])
    print("사용 재료:", result["used_ingredients"])
    print("추가 재료:", result["additional_ingredients"])
    print("사용하지 않은 재료:", result["unused_ingredients"])
    print("예상 시간:", result["recipe"]["estimated_time"])
    print("난이도:", result["recipe"]["difficulty"])
    print("추천 이유:", result["why_recommended"])


===== 테스트 1 =====
입력 재료: ['계란', '대파', '밥', '김치']
추천 요리: 계란 대파 볶음밥
한 줄 설명: 계란과 대파를 활용한 간단하고 맛있는 볶음밥 요리입니다.
사용 재료: ['계란', '대파', '밥']
추가 재료: ['소금', '후추', '식용유', '간장']
사용하지 않은 재료: [{'ingredient': '김치', 'reason': '김치의 강한 맛이 계란 대파 볶음밥과 어울리기 어렵고, 깔끔한 맛을 위해 제외함'}]
예상 시간: 15분
난이도: 쉬움
추천 이유: 냉장고 재료 중 자연스럽게 어울리는 계란, 대파, 밥을 사용해 간단하고 빠르게 만들 수 있는 가정식 볶음밥이기 때문입니다.

===== 테스트 2 =====
입력 재료: ['닭가슴살', '양배추', '당근', '양파']
추천 요리: 닭가슴살 야채볶음
한 줄 설명: 닭가슴살과 양배추, 당근, 양파를 함께 볶아 만든 건강한 한 끼 요리입니다.
사용 재료: ['닭가슴살', '양배추', '당근', '양파']
추가 재료: ['소금', '후추', '식용유', '간장', '다진 마늘']
사용하지 않은 재료: []
예상 시간: 20분
난이도: 쉬움
추천 이유: 입력된 모든 재료가 자연스럽게 어울리며 초보자도 쉽게 만들 수 있는 건강한 반찬으로 적합하기 때문입니다.

===== 테스트 3 =====
입력 재료: ['참치캔', '김치', '밥', '계란']
추천 요리: 참치김치볶음밥
한 줄 설명: 참치와 김치를 활용해 간단하게 만드는 매콤한 볶음밥
사용 재료: ['참치캔', '김치', '밥', '계란']
추가 재료: ['식용유', '간장', '소금', '후추']
사용하지 않은 재료: []
예상 시간: 20분
난이도: 쉬움
추천 이유: 참치, 김치, 밥, 계란이 서로 잘 어울리고 간단한 기본 양념으로 빠르게 만들 수 있어 가정식 한 끼로 적합하기 때문입니다.

===== 테스트 4 =====
입력 재료: ['파스타면', '베이컨', '마늘', '양파']
추천 요리: 베이컨 마늘 파